In [7]:
import fsspec
import xarray as xr
import zarr
#import zstd
import shutil
import os

In [3]:
fs = fsspec.filesystem("")

In [4]:
fs.glob('data/*abj*')

['/Users/peter/Local_Documents/heat_indices/UTCI_version/data/abj.zarr',
 '/Users/peter/Local_Documents/heat_indices/UTCI_version/data/abj_era5_500hpa_vars_1970_2025.nc',
 '/Users/peter/Local_Documents/heat_indices/UTCI_version/data/abj_utci.zarr',
 '/Users/peter/Local_Documents/heat_indices/UTCI_version/data/abj_utci_era5_winds.zarr']

In [9]:
ds.utci

<xarray.DataArray 'utci' (time: 18991, lat: 61, lon: 61)> Size: 565MB
dask.array<xarray-utci, shape=(18991, 61, 61), dtype=float64, chunksize=(1000, 30, 30), chunktype=numpy.ndarray>
Coordinates:
  * lon      (lon) float64 488B 10.5 10.75 11.0 11.25 ... 24.75 25.0 25.25 25.5
  * lat      (lat) float64 488B -25.6 -25.85 -26.1 -26.35 ... -40.1 -40.35 -40.6
  * time     (time) datetime64[ns] 152kB 1970-01-01T12:00:00 ... 2022-12-30T1...

In [14]:
for loc in ['cpt', 'jhb', 'abj']:
    ds = xr.open_dataset(f'/Users/peter/Local_Documents/heat_indices/UTCI_version/data/{loc}.zarr')
    ds_500 = xr.open_dataset(f'/Users/peter/Local_Documents/heat_indices/UTCI_version/data/{loc}_era5_500hpa_vars_1970_2025.nc')
    ds_500 = ds_500.isel(pressure_level = 0).drop_vars('pressure_level').rename({'valid_time':'time', 'latitude':'lat', 'longitude':'lon'}).rename({'t':'t_500','z':'z_500'})
    ds_500 = ds_500.sel(time = ds.time)
    ds_500 = ds_500.chunk(time = 1000, lat = 30, lon = 30)
    ds['z_500'] = ds_500.z_500
    ds['t_500'] = ds_500.t_500
    ds = ds.load()
    ds = ds.chunk(time = 1000, lat = 30, lon = 30)

    store_path = f'/Users/peter/Local_Documents/heat_indices/UTCI_version/data/{loc}_sfc_500.zarr'
    if os.path.exists(store_path):
        shutil.rmtree(store_path)
    encoding = {
                    var: {'chunks': tuple(dim_chunks[0] for dim_chunks in ds[var].chunks)}
                    for var in ds.data_vars
                }
    ds.to_zarr(store_path, encoding = encoding, mode='w', consolidated=True)

/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [50]:
ds

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 18991, lat: 61, lon: 61)
Coordinates:
  * lon      (lon) float64 488B 10.5 10.75 11.0 11.25 ... 24.75 25.0 25.25 25.5
  * lat      (lat) float64 488B -25.6 -25.85 -26.1 -26.35 ... -40.1 -40.35 -40.6
  * time     (time) datetime64[ns] 152kB 1970-01-01T12:00:00 ... 2022-12-30T1...
Data variables:
    utci     (time, lat, lon) float64 565MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>
    v10      (time, lat, lon) float32 283MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>
    u10      (time, lat, lon) float32 283MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>
    z_500    (time, lat, lon) float32 283MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>
    t_500    (time, lat, lon) float32 283MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>

In [44]:
ds.to_zarr('/Users/peter/Local_Documents/heat_indices/UTCI_version/data/cpt_sfc_500.zarr')

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 18991, lat: 61, lon: 61)
Coordinates:
  * lon      (lon) float64 488B 10.5 10.75 11.0 11.25 ... 24.75 25.0 25.25 25.5
  * lat      (lat) float64 488B -25.6 -25.85 -26.1 -26.35 ... -40.1 -40.35 -40.6
  * time     (time) datetime64[ns] 152kB 1970-01-01T12:00:00 ... 2022-12-30T1...
Data variables:
    utci     (time, lat, lon) float64 565MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>
    v10      (time, lat, lon) float32 283MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>
    u10      (time, lat, lon) float32 283MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>
    z_500    (time, lat, lon) float32 283MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>
    t_500    (time, lat, lon) float32 283MB dask.array<chunksize=(1000, 30, 30), meta=np.ndarray>

In [ ]:
import cdsapi
import os

def fetch_utci_by_decade(city_key, city_info, start_year=1970, end_year=2025, output_dir="."):
    client = cdsapi.Client()

    # Make decade ranges
    decades = [(y, min(y + 4, end_year)) for y in range(start_year, end_year + 1, 5)]
    months = [f"{m:02d}" for m in range(1, 13)]
    days = [f"{d:02d}" for d in range(1, 32)]

    for decade_start, decade_end in decades:
        years = [str(y) for y in range(decade_start, decade_end + 1)]
        outfile = os.path.join(output_dir, f"{city_key}_utci_{decade_start}_{decade_end}.nc")

        if os.path.exists(outfile):
            print(f"✅ Already exists: {outfile}")
            continue

        print(f"🔄 Requesting UTCI for {city_key}: {decade_start}–{decade_end}...")

        try:
            client.retrieve(
                "derived-utci-historical",
                {
                    "variable": "universal_thermal_climate_index",
                    "version": "1_1",
                    "product_type": "consolidated_dataset",
                    "year": years,
                    "month": months,
                    "day": days,
                    "area": city_info["area"],
                    "format": "netcdf",
                },
                outfile
            )
            print(f"✅ Finished: {outfile}")
        except Exception as e:
            print(f"❌ Failed for {city_key} {decade_start}–{decade_end}: {e}")

# Cities
cities = {
    "cpt": {"name": "Cape Town", "area": [-25.6, 10.5, -40.6, 25.5]},
    "jhb": {"name": "Johannesburg", "area": [-18.6, 20.5, -33.6, 35.5]},
    "abidjan": {"name": "Abidjan", "area": [12.8, -11.4, -2.2, 3.6]},
    "luanda": {"name": "Luanda", "area": [-4.0, 10.0, -13.5, 17.0]},  # rough bbox around Luanda
}


# 🏁 Start with Cape Town
#fetch_utci_by_decade("cpt", cities["cpt"])
#fetch_utci_by_decade("jhb", cities["jhb"])
#fetch_utci_by_decade("abidjan", cities["abidjan"])

fetch_utci_by_decade("luanda", cities["luanda"])
# must add .zip to all file names = mistake

2025-07-24 09:31:07,412 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.


✅ Already exists: ./luanda_utci_1970_1974.nc
🔄 Requesting UTCI for luanda: 1975–1979...


2025-07-24 09:31:08,389 INFO Request ID is 719c63f0-16aa-46cb-8946-96316de9f0bc
2025-07-24 09:31:08,805 INFO status has been updated to accepted
2025-07-24 09:31:23,229 WARNING Download format not supported for this dataset. Defaulting to zip.
2025-07-24 09:31:23,230 INFO status has been updated to running


In [2]:
import cdsapi
import os

def fetch_era5_winds_by_decade(city_key, city_info, start_year=1970, end_year=2025, output_dir="."):
    client = cdsapi.Client()

    # Make decade ranges
    decades = [(y, min(y + 4, end_year)) for y in range(start_year, end_year + 1, 5)]
    months = [f"{m:02d}" for m in range(1, 13)]
    days = [f"{d:02d}" for d in range(1, 32)]
    times = ["12:00"]

    for decade_start, decade_end in decades:
        years = [str(y) for y in range(decade_start, decade_end + 1)]
        outfile = os.path.join(output_dir, f"{city_key}_era5_winds_10m_{decade_start}_{decade_end}.nc.zip")

        if os.path.exists(outfile):
            print(f"✅ Already exists: {outfile}")
            continue

        print(f"🔄 Requesting ERA5 10m winds for {city_key}: {decade_start}–{decade_end}...")

        try:
            client.retrieve(
                "reanalysis-era5-single-levels",
                {
                    "product_type": "reanalysis",
                    "variable": [
                        "10m_u_component_of_wind",
                        "10m_v_component_of_wind",
                    ],
                    "year": years,
                    "month": months,
                    "day": days,
                    "time": times,
                    "area": city_info["area"],  # [North, West, South, East]
                    "format": "netcdf",
                },
                outfile
            )
            print(f"✅ Finished: {outfile}")
        except Exception as e:
            print(f"❌ Failed for {city_key} {decade_start}–{decade_end}: {e}")

# Cities
cities = {
    "cpt": {"name": "Cape Town", "area": [-25.6, 10.5, -40.6, 25.5]},
    "jhb": {"name": "Johannesburg", "area": [-18.6, 20.5, -33.6, 35.5]},
    "abidjan": {"name": "Abidjan", "area": [12.8, -11.4, -2.2, 3.6]},
    "luanda": {"name": "Luanda", "area": [-4.0, 10.0, -13.5, 17.0]},  # rough bbox around Luanda
}

# Fetch winds for all cities
#fetch_era5_winds_by_decade("cpt", cities["cpt"])
#fetch_era5_winds_by_decade("jhb", cities["jhb"])
#fetch_era5_winds_by_decade("abidjan", cities["abidjan"])
fetch_era5_winds_by_decade("luanda", cities["luanda"])

2025-07-08 16:25:32,254 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.


🔄 Requesting ERA5 10m winds for luanda: 1970–1974...


2025-07-08 16:25:33,299 INFO Request ID is e04cd64b-e08d-4077-b669-b01f17c28e45
2025-07-08 16:25:34,016 INFO status has been updated to accepted
2025-07-08 16:25:43,737 INFO status has been updated to running
2025-07-08 16:31:56,763 INFO status has been updated to successful


74dae3d5c7b4649dad1144f80bc9d611.nc:   0%|          | 0.00/9.29M [00:00<?, ?B/s]

✅ Finished: ./luanda_era5_winds_10m_1970_1974.nc.zip
🔄 Requesting ERA5 10m winds for luanda: 1975–1979...


...

6207000e1be867b8b1b22cc9f6b8fc9e.nc:   0%|          | 0.00/9.25M [00:00<?, ?B/s]

✅ Finished: ./luanda_era5_winds_10m_2020_2024.nc.zip
🔄 Requesting ERA5 10m winds for luanda: 2025–2025...


2025-07-08 17:35:25,302 INFO Request ID is a7f20bb1-2bd4-4cc8-ad53-0718997a28b4
2025-07-08 17:35:25,593 INFO status has been updated to accepted
2025-07-08 17:35:31,128 INFO status has been updated to running
2025-07-08 17:35:47,917 INFO status has been updated to accepted
2025-07-08 17:35:59,594 INFO status has been updated to running
2025-07-08 17:36:16,943 INFO status has been updated to successful


cdf7f28f926ce2a8348c3f7f3d5bf725.nc:   0%|          | 0.00/985k [00:00<?, ?B/s]

✅ Finished: ./luanda_era5_winds_10m_2025_2025.nc.zip


In [3]:
import cdsapi
import os

def fetch_era5_vars_500hpa_by_decade(city_key, city_info, start_year=1970, end_year=2025, output_dir="."):
    client = cdsapi.Client()

    decades = [(y, min(y + 4, end_year)) for y in range(start_year, end_year + 1, 5)]
    months = [f"{m:02d}" for m in range(1, 13)]
    days = [f"{d:02d}" for d in range(1, 32)]
    times = ["12:00"]

    for decade_start, decade_end in decades:
        years = [str(y) for y in range(decade_start, decade_end + 1)]
        outfile = os.path.join(output_dir, f"{city_key}_era5_500hpa_vars_{decade_start}_{decade_end}.nc.zip")

        if os.path.exists(outfile):
            print(f"✅ Already exists: {outfile}")
            continue

        print(f"🔄 Requesting ERA5 data at 500 hPa for {city_key}: {decade_start}–{decade_end}...")

        try:
            client.retrieve(
                "reanalysis-era5-pressure-levels",
                {
                    "product_type": "reanalysis",
                    "pressure_level": ["500"],
                    "variable": [
                        "geopotential",
                        "temperature"
                    ],
                    "year": years,
                    "month": months,
                    "day": days,
                    "time": times,
                    "area": city_info["area"],
                    "format": "netcdf",
                },
                outfile
            )
            print(f"✅ Finished: {outfile}")
        except Exception as e:
            print(f"❌ Failed for {city_key} {decade_start}–{decade_end}: {e}")

# Cities
cities = {
    "cpt": {"name": "Cape Town", "area": [-25.6, 10.5, -40.6, 25.5]},
    "jhb": {"name": "Johannesburg", "area": [-18.6, 20.5, -33.6, 35.5]},
    "abidjan": {"name": "Abidjan", "area": [12.8, -11.4, -2.2, 3.6]},
    "luanda": {"name": "Luanda", "area": [-4.0, 10.0, -13.5, 17.0]},
}

# Fetch for Luanda
fetch_era5_vars_500hpa_by_decade("luanda", cities["luanda"])
fetch_era5_vars_500hpa_by_decade("cpt", cities["cpt"])
fetch_era5_vars_500hpa_by_decade("abidjan", cities["abidjan"])
fetch_era5_vars_500hpa_by_decade("jhb", cities["jhb"])

2025-07-08 17:36:21,403 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.


🔄 Requesting ERA5 data at 500 hPa for luanda: 1970–1974...


2025-07-08 17:36:21,970 INFO Request ID is 6b32abfe-2923-424c-8b4b-44d4671cb845
2025-07-08 17:36:22,210 INFO status has been updated to accepted
2025-07-08 17:36:31,384 INFO status has been updated to running
2025-07-08 17:44:44,918 INFO status has been updated to successful


3f5e3cce4fbb86234dc9e2bb2a505034.nc:   0%|          | 0.00/5.80M [00:00<?, ?B/s]

✅ Finished: ./luanda_era5_500hpa_vars_1970_1974.nc.zip
🔄 Requesting ERA5 data at 500 hPa for luanda: 1975–1979...


...

2101db08df7b7b36edef04b2fd6648f3.nc:   0%|          | 0.00/27.0M [00:00<?, ?B/s]

✅ Finished: ./jhb_era5_500hpa_vars_2020_2024.nc.zip
🔄 Requesting ERA5 data at 500 hPa for jhb: 2025–2025...


2025-07-08 22:51:37,118 INFO Request ID is 06a8fa77-4a2f-4862-b965-df264a1f4ad4
2025-07-08 22:51:37,365 INFO status has been updated to accepted
2025-07-08 22:51:46,930 INFO status has been updated to running
2025-07-08 22:52:29,080 INFO status has been updated to successful


110a65d0c853387c29b2da87ced050af.nc:   0%|          | 0.00/2.33M [00:00<?, ?B/s]

✅ Finished: ./jhb_era5_500hpa_vars_2025_2025.nc.zip


In [17]:
import xarray as xr
import glob

# Pattern for your files
pattern = "*_v1.1_con.area-subset.-18.6.35.5.-33.6.20.5.nc"	
file_list = sorted(glob.glob(pattern))

# Open and concatenate along time dimension
ds = xr.open_mfdataset(file_list)

In [19]:
ds = ds.chunk({'time': 1000, 'lat': 30, 'lon': 30})

In [20]:
ds = ds.resample(time = '1D').max()

In [21]:
ds.to_netcdf('utci_con.area-subset.-18.6.35.5.-33.6.20.5.nc')

In [38]:
from numcodecs import Blosc

compressor = Blosc(cname="zstd", clevel=3, shuffle=2)

In [41]:
# ABJ
file_list = sorted(glob.glob('abidjan*'))
ds = xr.open_mfdataset(file_list)
ds_wind = ds.chunk({'valid_time': 1000, 'latitude': 30, 'longitude': 30})
ds_wind = ds_wind.drop('expver').drop('number').rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_wind.attrs = {}
ds_utci = xr.open_mfdataset('utci_con.area-subset.12.8.3.6.-2.2.-11.4.nc')
ds_wind['utci'] = ds_utci['utci']
ds = ds_wind.chunk({'time': 1000, 'lat': 30, 'lon': 30})
ds.to_zarr("abj_utci_era5_winds.zarr", mode='w', consolidated=True)

/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_5992/3954383374.py:5: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds_wind = ds_wind.drop('expver').drop('number').rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_5992/3954383374.py:10: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1975-01-01'. Serializing with units 'hours since 1975-01-01' instead. Set encoding['dtype'] to floating point dtype to serialize with units 'days since 1975-01-01'. Set encoding['units'] to 'hours since 1975-01-01' to silence this warning .
  ds.to_zarr("abj_utci_era5_winds.zarr", mode='w', consolidated=True)
/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by 

In [43]:
# JHB
file_list = sorted(glob.glob('jhb*'))
ds = xr.open_mfdataset(file_list)
ds_wind = ds.chunk({'valid_time': 1000, 'latitude': 30, 'longitude': 30})
ds_wind = ds_wind.drop('expver').drop('number').rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_wind.attrs = {}
ds_utci = xr.open_mfdataset('utci_con.area-subset.-18.6.35.5.-33.6.20.5.nc')
ds_wind['utci'] = ds_utci['utci']



/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_5992/370979334.py:5: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds_wind = ds_wind.drop('expver').drop('number').rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_5992/370979334.py:10: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1970-01-01'. Serializing with units 'hours since 1970-01-01' instead. Set encoding['dtype'] to floating point dtype to serialize with units 'days since 1970-01-01'. Set encoding['units'] to 'hours since 1970-01-01' to silence this warning .
  ds.to_zarr("jhb_utci_era5_winds.zarr", mode='w', consolidated=True)
/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by ot

In [44]:
# cpt
file_list = sorted(glob.glob('cpt*'))
ds = xr.open_mfdataset(file_list)
ds_wind = ds.chunk({'valid_time': 1000, 'latitude': 30, 'longitude': 30})
ds_wind = ds_wind.drop('expver').drop('number').rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_wind.attrs = {}
ds_utci = xr.open_mfdataset('utci_con.area-subset.-25.6.25.5.-40.6.10.5.nc')
ds_wind['utci'] = ds_utci['utci']
ds = ds_wind.chunk({'time': 1000, 'lat': 30, 'lon': 30})
ds.to_zarr("cpt_utci_era5_winds.zarr", mode='w', consolidated=True)

/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_5992/1519540507.py:5: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds_wind = ds_wind.drop('expver').drop('number').rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_5992/1519540507.py:10: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1970-01-01'. Serializing with units 'hours since 1970-01-01' instead. Set encoding['dtype'] to floating point dtype to serialize with units 'days since 1970-01-01'. Set encoding['units'] to 'hours since 1970-01-01' to silence this warning .
  ds.to_zarr("cpt_utci_era5_winds.zarr", mode='w', consolidated=True)
/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by 

In [38]:
import xarray as xr
import zarr
import pandas as pd
import numpy as np


In [47]:
ds = xr.open_zarr('data/cpt_utci_era5_winds.zarr')
ds_utci = xr.open_zarr('data/cpt_utci.zarr').rename({'latitude': 'lat', 'longitude': 'lon'})
ds_utci = ds_utci.interp(lat = ds.lat, lon = ds.lon).UTCI.load()
ds = ds.sel(time = slice(ds_utci.time[0], ds_utci.time[-1]))
ds = ds.load()
ds_utci["time"] = xr.DataArray(
    [pd.Timestamp(t).replace(hour=12, minute=0) if pd.Timestamp(t).hour == 11 and pd.Timestamp(t).minute == 30 else pd.Timestamp(t)
     for t in ds_utci.time.values],
    dims="time"
)
ds["time"] = xr.DataArray(
    [pd.Timestamp(t).replace(hour=12, minute=0) if pd.Timestamp(t).hour == 11 and pd.Timestamp(t).minute == 30 else pd.Timestamp(t)
     for t in ds.time.values],
    dims="time"
)
ds = ds.drop('utci')
common_times = np.intersect1d(ds_utci.time.values, ds.time.values)
ds_utci = ds_utci.sel(time=common_times)
ds = ds.sel(time=common_times)
ds['utci'] = (('time', 'lat', 'lon'), ds_utci.values)
ds = ds.chunk({'time': 1000, 'lat': 30, 'lon': 30}).load()
ds.to_zarr('data/cpt.zarr', mode='w', consolidated=True)


/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_34396/4211591838.py:16: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds = ds.drop('utci')
/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [48]:
ds = xr.open_zarr('data/jhb_utci_era5_winds.zarr')
ds_utci = xr.open_zarr('data/jhb_utci.zarr').rename({'latitude': 'lat', 'longitude': 'lon'})
ds_utci = ds_utci.interp(lat = ds.lat, lon = ds.lon).UTCI.load()
ds = ds.sel(time = slice(ds_utci.time[0], ds_utci.time[-1]))
ds = ds.load()
ds_utci["time"] = xr.DataArray(
    [pd.Timestamp(t).replace(hour=12, minute=0) if pd.Timestamp(t).hour == 11 and pd.Timestamp(t).minute == 30 else pd.Timestamp(t)
     for t in ds_utci.time.values],
    dims="time"
)
ds["time"] = xr.DataArray(
    [pd.Timestamp(t).replace(hour=12, minute=0) if pd.Timestamp(t).hour == 11 and pd.Timestamp(t).minute == 30 else pd.Timestamp(t)
     for t in ds.time.values],
    dims="time"
)
ds = ds.drop('utci')
common_times = np.intersect1d(ds_utci.time.values, ds.time.values)
ds_utci = ds_utci.sel(time=common_times)
ds = ds.sel(time=common_times)
ds['utci'] = (('time', 'lat', 'lon'), ds_utci.values)
ds = ds.chunk({'time': 1000, 'lat': 30, 'lon': 30}).load()
ds.to_zarr('data/jhb.zarr', mode='w', consolidated=True)

/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_34396/558859223.py:16: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds = ds.drop('utci')
/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [50]:
ds = xr.open_zarr('data/abj_utci_era5_winds.zarr')
ds_utci = xr.open_zarr('data/abidjan_utci.zarr').rename({'latitude': 'lat', 'longitude': 'lon'})
ds_utci = ds_utci.interp(lat = ds.lat, lon = ds.lon).UTCI.load()
ds = ds.sel(time = slice(ds_utci.time[0], ds_utci.time[-1]))
ds = ds.load()
ds_utci["time"] = xr.DataArray(
    [pd.Timestamp(t).replace(hour=12, minute=0) if pd.Timestamp(t).hour == 11 and pd.Timestamp(t).minute == 30 else pd.Timestamp(t)
     for t in ds_utci.time.values],
    dims="time"
)
ds["time"] = xr.DataArray(
    [pd.Timestamp(t).replace(hour=12, minute=0) if pd.Timestamp(t).hour == 11 and pd.Timestamp(t).minute == 30 else pd.Timestamp(t)
     for t in ds.time.values],
    dims="time"
)
ds = ds.drop('utci')
common_times = np.intersect1d(ds_utci.time.values, ds.time.values)
ds_utci = ds_utci.sel(time=common_times)
ds = ds.sel(time=common_times)
ds['utci'] = (('time', 'lat', 'lon'), ds_utci.values)
ds = ds.chunk({'time': 1000, 'lat': 30, 'lon': 30}).load()
ds.to_zarr('data/abj.zarr', mode='w', consolidated=True)

/var/folders/dh/9zq64sxs3dld38sgl5gmq_4h0000gn/T/ipykernel_34396/2480484049.py:16: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds = ds.drop('utci')
/Users/peter/.local/share/mamba/envs/som_pytorch_env/lib/python3.12/site-packages/zarr/api/asynchronous.py:205: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
